# Sistem Pendukung Keputusan untuk Membantu Pengguna Menentukan Kendaraan Listrik Terbaik untuk Penggunaan Sehari-hari

In [1]:
# Import Library yang diperlukan
import pandas as pd
import numpy as np
import streamlit as st


In [2]:
# Function
def weightNormalization(w):
    w = np.array(w)
    return w / np.sum(w)

In [3]:
# Ekstrak Dataset
dataFrame = pd.read_csv("default_ev_spec_dataset.csv")

newDataFrame = dataFrame[['brand', 'model', 'range_km', 'efficiency_wh_per_km', 'acceleration_0_100_s', 'fast_charging_power_kw_dc', 'seats', 'cargo_volume_l']]

newDataFrame.to_csv("clean_ev_spec_dataset.csv", index=False)


In [4]:
data = pd.read_csv('clean_ev_spec_dataset.csv')
print("Jumlah data sebelum cleaning:", len(data))

criteriaColumns = [
    'range_km', #Benefit
    'efficiency_wh_per_km', #Cost
    'acceleration_0_100_s', #Cost
    'fast_charging_power_kw_dc', #Benefit
    'seats', #Cost
    'cargo_volume_l' #Benefit
]


dataNumeric = data[criteriaColumns].apply(pd.to_numeric, errors='coerce')

dataCleaned = data[dataNumeric.notna().all(axis=1)].copy()
dataCleaned[criteriaColumns] = dataNumeric[dataNumeric.notna().all(axis=1)]

dataCleaned = dataCleaned[(dataCleaned[criteriaColumns] != 0).all(axis=1)]

print("Jumlah data setelah cleaning:", len(dataCleaned))


dataFrame = dataCleaned[['brand', 'model'] + criteriaColumns].reset_index(drop=True)

Jumlah data sebelum cleaning: 478
Jumlah data setelah cleaning: 473


In [5]:
# Normalisasi Bobot
criteriaWeight = np.array([1/6] * 6)
# print(criteriaWeight)

normalizeWeight = weightNormalization(criteriaWeight)
normalizeWeight[[1, 2]] *= -1 #Normalisasi bobot untuk cost

print(normalizeWeight)

[ 0.16666667 -0.16666667 -0.16666667  0.16666667  0.16666667  0.16666667]


In [ ]:
# Hitung nilai S

S = []

for i in range(len(dataFrame)):
    nilai = 1
    for j, col in enumerate(criteriaColumns):
        nilai *= dataFrame.loc[i, col] ** normalizeWeight[j]
    S.append(nilai)

dataFrame["S"] = S


In [7]:
# Hitung Nilai V (preferensi)
total_S = dataFrame['S'].sum()
dataFrame['V'] = dataFrame['S'] / total_S

In [8]:
# Perankingan
dataFrame['Rank'] = dataFrame['V'].rank(ascending=False)

dataFrameFinal = dataFrame.sort_values('V', ascending=False).reset_index(drop=True)

print(dataFrameFinal.head())


     brand                         model  range_km  efficiency_wh_per_km  \
0    Tesla                 Model S Plaid       560                   158   
1  Porsche  Taycan Turbo S Sport Turismo       505                   183   
2    Lucid             Air Grand Touring       665                   143   
3  Porsche                Taycan Turbo S       525                   174   
4    Tesla            Model S Dual Motor       575                   150   

   acceleration_0_100_s  fast_charging_power_kw_dc  seats  cargo_volume_l  \
0                   2.3                      140.0      5           709.0   
1                   2.4                      281.0      5           405.0   
2                   3.0                      184.0      5           456.0   
3                   2.4                      281.0      5           366.0   
4                   3.2                      140.0      5           709.0   

          S         V  Rank  
0  9.562936  0.002920   1.0  
1  9.317371  0.00284